### Evaluation du modèle sur les messages anglais
Ce notebook teste le modèle de Matous et al. sur nos messages annotés en anglais

In [ ]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import langid
import numpy as np
from torch import argmax

### Récupération des annotations en anglais

In [ ]:
df = pd.read_csv("../annotations_left-right/sample_annotated_left-right.csv")
df = df[df['annotation']!='Unclassifiable']

langid.set_languages(['en', 'fr'])
df["language"] = df["text"].apply(lambda x: langid.classify(x)[0])
df = df[df['language']=='en']
texts = df["text"].astype(str).tolist()
labels = df['annotation'].astype(str).tolist()
y_test = [0 if l=='Left' else 1 for l in labels]

In [ ]:
N_CHUNKS = 20
parts = np.array_split(texts, N_CHUNKS)
print(f"Chunk sizes: {[len(p) for p in parts]}")

### Chargement du modèle

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-large', use_fast=False)

model = AutoModelForSequenceClassification.from_pretrained(
    "matous-volf/political-leaning-deberta-large"
    ).to(device)

In [ ]:
model.eval()

max_length = model.config.max_position_embeddings

### Inférence

In [ ]:
y_pred = []
with torch.no_grad():
    for i, chunk in enumerate(parts):
        encodings = tokenizer(
            chunk.tolist(),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encodings = {k: v.to(device) for k, v in encodings.items()}

        outputs = model(**encodings)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        y_pred.extend(preds)

### Calcul des métriques

In [ ]:
# Métriques par classe
precision_per_class = precision_score(y_test, y_pred, average=None)
recall_per_class = recall_score(y_test, y_pred, average=None)
f1_per_class = f1_score(y_test, y_pred, average=None)

# Support par classe
support_class_0 = len(y_test) - sum(y_test)
support_class_1 = sum(y_test)

# Moyennes macro
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Affichage
for i in range(2):
    support = support_class_0 if i == 0 else support_class_1

    print(f"Classe {i}")
    print(f"Precision : {precision_per_class[i]:.4f}")
    print(f"Recall    : {recall_per_class[i]:.4f}")
    print(f"F1-score  : {f1_per_class[i]:.4f}")
    print(f"Support   : {support}")
    print()

print("=== Moyennes macro ===")
print(f"Precision macro : {precision_macro:.4f}")
print(f"Recall macro    : {recall_macro:.4f}")
print(f"F1 macro        : {f1_macro:.4f}")

print("\n=== Accuracy ===")
print(f"Accuracy : {accuracy:.4f}")